# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi
14036,113361,THE FAMILY CONVENIENCE INC,3104 36TH AVE,ASTORIA,11106,independent,CONV. STORE,-73.92954,40.75648,b'\x01\x01\x00\x00\x00\xba\x14W\x95}{R\xc0\xff\t.V\xd4`D@',360810051001,36081005100,118
16172,116380,PERFETTA CORP,777 W BEECH ST,LONG BEACH,11561,independent,RESTAURANT-HOTEL,-73.68665,40.58602,b'\x01\x01\x00\x00\x00\x19s\xd7\x12\xf2kR\xc0\xa5f\x0f\xb4\x02KD@',360594164022,36059416402,12
11265,107008,LAKE VILLAGE MARKET,904 JOHNSON AVE,RONKONKOMA,11779,independent,SUPERMARKETS,-73.12292,40.80725,b'\x01\x01\x00\x00\x00\x8e\x01\xd9\xeb\xddGR\xc0\x17\xd9\xce\xf7SgD@',361031458082,36103145808,16
6535,69224,QUICKLEES #05,3 MAIN ST,BLOOMFIELD,14469,chain,CONV. STORE,-77.42065,42.90231,b'\x01\x01\x00\x00\x00\xfeC\xfa\xed\xebZS\xc0Hm\xe2\xe4~sE@',360690505012,36069050501,2
3810,26400,BOTTLE BARGAINS,1033 FORT SALONGA RD,NORTHPORT,11768,independent,LIQUOR,-73.31481,40.90825,b'\x01\x01\x00\x00\x00\n\x9d\xd7\xd8%TR\xc0\xc7K7\x89AtD@',361031108032,36103110803,2


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
4547,112459,880918.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,2413.473973,2413.473973,False,2413.473973
2998,109486,159827.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,352.0,365,365.0,0.0,0.0,437.882192,437.882192,False,437.882192
3092,109642,547953.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,1501.241096,1501.241096,False,1501.241096
11551,74088,647191.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1773.126027,1773.126027,False,1773.126027
4561,112476,50734.5,2025-01-01,2025-10-22,2025-01-01,2025-03-21,80.0,295,80.0,0.0,215.0,171.981356,634.181250,True,634.181250


## Overture Maps Data

In [7]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df['highest_hierarchy'] = interim_overture_df['taxonomy_hierarchy'].str[0]
print(interim_overture_df['highest_hierarchy'].unique())
print(interim_overture_df['basic_category'].nunique())
interim_overture_df.sample(20)

['services_and_business' 'sports_and_recreation' 'education' 'health_care'
 'arts_and_entertainment' 'shopping' 'lifestyle_services'
 'travel_and_transportation' 'community_and_government'
 'cultural_and_historic' 'geographic_entities' 'food_and_drink' 'lodging'
 None]
201


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen,highest_hierarchy
2956,d6b59372-31cc-4b6d-93c0-180f9a2e14c7,b'\x01\x01\x00\x00\x00\xc7q\x85\xabe/R\xc03\xf2\xc4\xaa\xaazD@',-72.740580,40.958333,golf_course,golf_course,"[sports_and_recreation, sport_or_fitness_facility, golf_course]",0.990219,NaN,2025-06-25.0,sports_and_recreation
1737,1c7433f2-a8fd-436b-8d3a-c0aa89e4fcd3,"b'\x01\x01\x00\x00\x00\xe1)?\x04\xb0.R\xc0""\xc8\x9c\xc2\xa5iD@'",-72.729493,40.825371,second_hand_store,antique_store,"[shopping, second_hand_store, antique_store]",0.919912,NaN,2025-06-25.0,shopping
4235,9e60afb1-967c-48a8-b817-0ccca2cf731a,b'\x01\x01\x00\x00\x00\xb7E\x99\r2wR\xc0\x98\x86\xe1#bZE@',-73.862430,42.706120,civic_organization,charity_organization,"[community_and_government, civic_organization, charity_organization]",0.919912,NaN,2025-06-25.0,community_and_government
387,72d8ea4c-622a-44d6-a27d-2a17cc3e8c20,b'\x01\x01\x00\x00\x00/\xdd$\x06\x81-R\xc0\xaf%\xe4\x83\x9euD@',-72.711000,40.918900,sporting_goods_store,sportswear_store,"[shopping, specialty_store, sporting_goods_store, sportswear_store]",0.991963,NaN,2025-06-25.0,shopping
4777,5961ccc3-d33e-42a0-bf3c-95623b03899c,b'\x01\x01\x00\x00\x00\x90\xb8*\xfdBuR\xc0Q\xa2%\x8f\xa7OE@',-73.832214,42.622301,bank_or_credit_union,bank,"[services_and_business, financial_service, bank_or_credit_union, bank]",0.999022,KeyBank,2025-06-25.0,services_and_business
4083,80e9fc73-5f48-4faa-b994-4ff48ea41739,b'\x01\x01\x00\x00\x00\x05\xb90wRwR\xc0\xf3{\xf6\xb7_\\E@',-73.864408,42.721671,professional_service,janitorial_service,"[services_and_business, professional_service, cleaning_service, janitorial_service]",0.919912,NaN,2025-06-25.0,services_and_business
1455,d3dcd312-112e-4211-aacb-662ba2dd41ab,b'\x01\x01\x00\x00\x00\xf4\xa8\xf8\xbf#)R\xc04\xbd\xc4X\xa6gD@',-72.642807,40.809764,fashion_and_apparel_store,clothing_store,"[shopping, fashion_and_apparel_store, clothing_store]",0.919912,NaN,2025-06-25.0,shopping
3966,eb084b9f-0e25-4041-9790-34861a932957,b'\x01\x01\x00\x00\x00\xac%6\x82AxR\xc0b\xf7\xfc\xb4\xfaXE@',-73.878998,42.695151,physical_medicine_and_rehabilitation,physical_therapy,"[health_care, outpatient_care_facility, physical_medicine_and_rehabilitation, physical_therapy]",0.919912,Core Physical Therapy,2025-06-25.0,health_care
2782,388107b5-9ca8-40f3-944c-e314c5635b8f,b'\x01\x01\x00\x00\x00\x9b\xacQ\x0f\xd15R\xc0\x1a\x864\xcf\xb6xD@',-72.840885,40.943079,shopping,shopping,[shopping],0.990219,NaN,2025-06-25.0,shopping
2637,a03d496c-c2aa-458b-959b-c3dd005791b2,"b'\x01\x01\x00\x00\x00\xb0E\xc1B\x1c;R\xc0\x13,\x0eg~yD@'",-72.923600,40.949170,casual_eatery,delicatessen,"[food_and_drink, casual_eatery, delicatessen]",0.975911,NaN,2025-06-25.0,food_and_drink


In [8]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [9]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,commercial_places_within_1_mi,community_places_within_1_mi
41,65525,SANDY'S CARD GIFT & STATIONERY,1355 E MONTAUK HWY #9,MASTIC,11950,0,STATIONERY-CARD,-72.85039,40.80482,b'\x01\x01\x00\x00\x00\x1c\xb6-\xcal6R\xc0W\x95}W\x04gD@',361031594164,36103159416,145,11
91,58065,FAST STOP FOOD MART,326 MILITARY RD,BUFFALO,14207,0,CONV. STORE,-78.89153,42.94573,b'\x01\x01\x00\x00\x00\xc9Y\xd8\xd3\x0e\xb9S\xc0Ll>\xae\ryE@',360290056003,36029005600,191,67
28,75940,EFE CORP OK PETROLEUM,767 WANTAGH AVE,WANTAGH,11793,0,GAS,-73.50327,40.70508,"b""\x01\x01\x00\x00\x00\xb5\xc3_\x935`R\xc0'\x88\xba\x0f@ZD@""",360594094003,36059409400,81,18
29,106093,QUICK START CONVENIENCE,465 ROUTE 17M,MIDDLETOWN,10940,0,CONV. STORE,-74.41335,41.46407,b'\x01\x01\x00\x00\x00M\xf3\x8eSt\x9aR\xc0\x01\x87P\xa5f\xbbD@',360710113013,36071011301,86,28
20,110259,ALMOST FREE,500 PEARL ST LOBBY,NEW YORK,10007,230,CONV. STORE,-74.00135,40.71355,b'\x01\x01\x00\x00\x00`vO\x1e\x16\x80R\xc0\xc8\x07=\x9bU[D@',360610029011,36061002901,4761,1099


## LEHD Data

In [10]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [11]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
63,78649,CT HEALTH CASTLE CORP,86 BOWERY,NEW YORK,10013,-73.99557,40.71705,b'\x01\x01\x00\x00\x00=\xb8;k\xb7\x7fR\xc0d\xcc]K\xc8[D@',360610041002,36061004100,533.0,93.0,119.0,321.0
48,70164,BASSI'S XPRESS INC,2543 ROUTE 302,MIDDLETOWN,10941,-74.40191,41.49438,b'\x01\x01\x00\x00\x00\xdd{\xb8\xe4\xb8\x99R\xc0\xf3\xe5\x05\xd8G\xbfD@',360710110003,36071011000,448.0,68.0,100.0,280.0


In [12]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [13]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
29,76146,RITE AID #10642,695 EAST JERICHO TURNPIKE,HUNTINGTON STATION,11746,-73.38412,40.83514,b'\x01\x01\x00\x00\x00To\rl\x95XR\xc0m\xca\x15\xde\xe5jD@',361031114021,36103111402,626.0,384.0,46.0,90.0,132.0,404.0
98,95155,ARIAS LIQUOR INC,1648 BRUCKNER BLVD,BRONX,10473,-73.87382,40.82335,b'\x01\x01\x00\x00\x00\xce\xa5\xb8\xaa\xecwR\xc0\xaf\x94e\x88ciD@',360050028001,36005002800,989.0,6.0,85.0,129.0,281.0,579.0


## AADT Data

In [14]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [15]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

9759 16801


,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
14442,112745,534 GRANT GROCERY INC,534 GRANT AVE,BROOKLYN,11208,-73.86579,40.67771,360471188002,36047118800,NaN,NaN,NaN,NaN,NaN,NaN
6749,78731,CASTLE FINANCIAL SERVICES,4173 BROADWAY,NEW YORK,10033,-73.93814,40.84727,360610263005,36061026300,54676.0,152.0,Depot Pl,19,1:0||||,943.961727
11424,78498,CITGO FOOD MART,650 COUNTY ROUTE 1,PINE ISLAND,10969,-74.46303,41.29658,360710145011,36071014501,NaN,NaN,NaN,NaN,NaN,NaN
13886,111006,7-ELEVEN STORE #40234,226 NORTH AVENUE,WEBSTER,14580,-77.43009,43.21880,360550114032,36055011403,NaN,NaN,NaN,NaN,NaN,NaN
13584,109836,KINGS 43 GLEN COVE C-STORE,43 GLEN COVE AVE,GLEN COVE,11542,-73.63367,40.86232,360595172011,36059517201,NaN,NaN,NaN,NaN,NaN,NaN
12619,104413,NOXON SUNOCO,349 NOXON RD,POUGHKEEPSIE,12603,-73.84783,41.65950,360270802011,36027080201,NaN,NaN,NaN,NaN,NaN,NaN
6962,105854,7-ELEVEN #40494,238 W MAIN ST,GOUVERNEUR,13642,-75.47648,44.33021,360894927005,36089492700,54327.0,45.0,SOUTH JOHN ST,09,1:1||||,796.663182
6588,59238,EASTOWN BEVERAGE,563 E MAIN ST,BATAVIA,14020,-78.16629,42.99610,360379509002,36037950900,54303.0,233.0,WEBSTER AVE,19,1:1||||,577.343486
10420,67781,WESTBURY MEAT CORP,595 OLD COUNTRY RD,WESTBURY,11590,-73.57551,40.75082,360593042023,36059304202,NaN,NaN,NaN,NaN,NaN,NaN
12811,105695,STOP & SHOP #2520,245 ROUTE 25A,ROCKY POINT,11778,-72.94811,40.94519,361031583183,36103158318,NaN,NaN,NaN,NaN,NaN,NaN


## EPA Data

In [16]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [17]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
0,55194,BOWL-O-DROME,2712 PINE AVE,NIAGARA FALLS,14301,0,BOWLING CENTERS,-79.028780,43.095130,b'\x01\x01\x00\x00\x00\xd9\xb1\x11\x88\xd7\xc1S\xc0*oG8-\x8cE@',360630210002,36063021000,149662,36,63,21000,2,6.676373,13.003563,1.941456,0.432987,0.502823,0.125706,0.223477,0.572660,0.000000,0.083804,0.000000,31.913523,17.333333,36,063,021000,2
2,85214,HONG KONG SUPERMARKET,37-11 MAIN STREET,FLUSHING,11354,0,SUPERMARKETS,-73.830940,40.761190,b'\x01\x01\x00\x00\x00\x0c<\xf7\x1e.uR\xc0h\x05\x86\xacnaD@',360810871002,36081087100,156383,36,81,87100,2,5.169308,11.224783,101.236384,10.338616,13.029938,8.156019,20.168506,22.695724,4.217499,22.630082,0.000000,24.236835,18.166667,36,081,087100,2
29,77453,JAS 109 MART INC,605 ROUTE 109,WEST BABYLON,11704,0,CONV. STORE,-73.376360,40.713580,b'\x01\x01\x00\x00\x00r\xe1@H\x16XR\xc03\x8a\xe5\x96V[D@',361031229022,36103122902,160519,36,103,122902,2,5.176989,12.282183,1.482612,0.769662,0.194441,0.064814,0.064814,0.072915,0.064814,0.251153,0.000000,30.881925,12.833333,36,103,122902,2
82,111083,MOCHA CAFE & GRIL CORP,300 STATE HIGHWAY 67,AMSTERDAM,12010,0,CONV. STORE,-74.144550,42.947050,b'\x01\x01\x00\x00\x00\xc1\xa8\xa4N@\x89R\xc0\xa1\xd64\xef8yE@',360570721002,36057072100,160713,36,57,72100,2,0.237802,0.471652,0.094377,0.002092,0.000697,0.006974,0.022781,0.002557,0.000000,0.047421,0.011855,3.186261,5.666667,36,057,072100,2
35,78975,BULLVILLE FOOD MART,2887 ROUTE 17K,BULLVILLE,10915,0,CONV. STORE,-74.358904,41.541798,"b'\x01\x01\x00\x00\x00\x1d\xcb""G\xf8\x96R\xc0wI\xdf\xa5Y\xc5D@'",360710109023,36071010902,153458,36,71,10902,3,0.134975,0.404333,0.059496,0.008880,0.000296,0.026048,0.016872,0.000000,0.000000,0.007400,0.000000,2.878447,5.166667,36,071,010902,3


## ACS Data

In [18]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [19]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
15,93307,SPEEDWAY #07827,810 METROPOLITAN AVE,BROOKLYN,11211,46,CONV. STORE,-73.94166,40.71472,b'\x01\x01\x00\x00\x00\xe3\xfcM(D|R\xc0\x08\xe6\xe8\xf1{[D@',360470477002,36047047700,198884.0,937.0,1761.0,0.164111,0.822828,0.013061,0.039409,0.579857,0.274136,0.942646,0.016867,0.160085,0.839915,0.410886,0.275347,0.313767,0.558164
95,112326,HARTSDALE TOBACCO & CANDY CORP,205 E HARTSDALE AVE,HARTSDALE,10530,0,CONV. STORE,-73.79610,41.01232,b'\x01\x01\x00\x00\x00\x1e\x16jM\xf3rR\xc0\x1a\x8b\xa6\xb3\x93\x81D@',361190108041,36119010804,108958.0,767.0,1447.0,0.070491,0.536973,0.392536,0.035932,0.352597,0.381858,0.618656,0.041020,0.885267,0.114733,0.629726,0.009126,0.361147,0.067797
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,0,GROCERY STORE,-73.88996,40.91546,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\xc0\xe9e\x14\xcb-uD@',361190013013,36119001301,69635.0,676.0,1402.0,0.206134,0.619116,0.174750,0.225919,0.059004,0.110345,0.588608,0.000000,0.289941,0.710059,0.779586,0.000000,0.220414,0.474852
80,99709,RITE AID #10534,225 LIBERTY STREET STE 1TER,NEW YORK,10281,296,DRUG STORE,-74.01544,40.71141,b'\x01\x01\x00\x00\x007\xc3\r\xf8\xfc\x80R\xc0\x10\x06\x9e{\x0f[D@',360610317041,36061031704,184291.0,1218.0,2334.0,0.257498,0.670951,0.071551,0.061022,0.556870,0.330755,0.873179,0.035819,0.000000,1.000000,0.274220,0.164204,0.561576,0.747947
85,114079,NEWTOWN CONVENIENT STORE INC,2931 NEWTOWN AVE,ASTORIA,11102,0,VARIETY,-73.92141,40.76855,b'\x01\x01\x00\x00\x00E\r\xa6a\xf8zR\xc0\x9f\xab\xad\xd8_bD@',360810069001,36081006900,152304.0,990.0,1825.0,0.254795,0.692055,0.053151,0.140109,0.309482,0.468296,0.794788,0.054645,0.080808,0.919192,0.416162,0.228283,0.355556,0.534343


## Master Data Sample

In [20]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(16740, 60)
Index(['agtno', 'busnm', 'busaddr', 'buscity', 'buszip', 'chain', 'bustype',
       'longitude', 'latitude', 'geometry', 'bg_geoid', 'tract_geoid',
       'competitors_within_1_mi', 'aadt_count', 'aadt_road_name',
       'aadt_functional_class', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'households_total', 'total_population',
       'age_18_29_proportion', 'age_30_64_proportion',
       'age_65_plus_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'renter_occupied_proportion', 'family_households_proportion',
       'nonfamily_multiperson_households_proportion',
       'living_alone_households_proportion',
       'no_vehicle_households_proportion', 'D1A', 'D1B', 'D1C', 'D1C8_RET',
       'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT', 'D1C8_ED', 'D1C8_HLTH',
       'D1C8_PUB', 'D3A', 'N

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,commercial_places_within_1_mi,community_places_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
14212,113663,YUNUS PETROLEUM INC,1309 ALTAMONT AVE,SCHENECTADY,12303,independent,CONV. STORE,-73.94347,42.78613,b'\x01\x01\x00\x00\x00u\xb0\xfe\xcfa|R\xc0\xc63h\xe8\x9fdE@',360930333001,36093033300,23,6007.0,Duane Ave,14,1:1|1:2|2:2||,1462.474330,74404.0,601.0,1046.0,0.274379,0.555449,0.170172,0.009709,0.143982,0.032621,0.646272,0.039941,0.803661,0.196339,0.554077,0.058236,0.387687,0.044925,1.887838,3.899776,2.124757,1.116908,0.146665,0.300851,0.094016,0.23692,0.011282,0.218117,0.0,18.357121,16.000000,134,30,710.0,97.0,187.0,426.0,619.0,339.0,128.0,191.0,233.0,195.0
13979,113269,CORNER MARKET,38235 STATE ROUTE 37,THERESA,13691,independent,CONV. STORE,-75.82167,44.22230,b'\x01\x01\x00\x00\x00\xad\x86\xc4=\x96\xf4R\xc0M\xf3\x8eSt\x1cF@',360450601012,36045060101,1,NaN,NaN,NaN,NaN,NaN,97708.0,352.0,624.0,0.307692,0.532051,0.160256,0.056728,0.043388,0.105372,0.752381,0.129590,0.815341,0.184659,0.477273,0.099432,0.423295,0.096591,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,1,205.0,32.0,60.0,113.0,65.0,36.0,0.0,14.0,30.0,21.0
3157,101895,MMP MEGAMART INC,1432 ROUTE 9D,WAPPINGERS FALLS,12590,independent,CONV. STORE,-73.96855,41.52511,b'\x01\x01\x00\x00\x00\x9c\xa2#\xb9\xfc}R\xc0\xbaf\xf2\xcd6\xc3D@',360270601001,36027060100,4,NaN,NaN,NaN,NaN,NaN,74153.0,999.0,1394.0,0.283357,0.465567,0.251076,0.084099,0.413205,0.100890,0.691535,0.097510,0.308308,0.691692,0.194194,0.108108,0.697698,0.024024,0.969952,2.921749,0.182362,0.011893,0.000000,0.040965,0.074002,0.01850,0.000000,0.037001,0.0,9.272500,6.666667,13,11,522.0,82.0,103.0,337.0,26.0,0.0,0.0,2.0,6.0,18.0
5122,46853,E-Z FOOD MARKET,700 S PLYMOUTH AVE,ROCHESTER,14608,independent,CONV. STORE,-77.61961,43.14205,b'\x01\x01\x00\x00\x00\x92\x91\xb3\xb0\xa7gS\xc0\xca2\xc4\xb1.\x92E@',360550065002,36055006500,18,2477.0,Woodbury Blvd,17,1:1||||,1539.976859,21023.0,440.0,679.0,0.265096,0.603829,0.131075,0.516876,0.053763,0.051971,0.601744,0.111111,0.152273,0.847727,0.502273,0.122727,0.375000,0.377273,8.506856,18.537072,3.560213,0.273863,0.000000,2.858440,0.410794,0.00000,0.000000,0.017116,0.0,26.009681,15.333333,183,121,315.0,78.0,143.0,94.0,233.0,18.0,1.0,15.0,142.0,76.0
11172,106544,NOR BRENTWOOD SERVICE CENTER,1490 ISLIP AVE,BRENTWOOD,11717,independent,CONV. STORE,-73.21576,40.77740,b'\x01\x01\x00\x00\x00=\xf2\x07\x03\xcfMR\xc0\x88\xf4\xdb\xd7\x81cD@',361031462012,36103146201,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.606249,15.589075,0.270019,0.042003,0.000000,0.024002,0.018001,0.00600,0.000000,0.180012,0.0,21.210591,7.166667,146,36,1310.0,185.0,488.0,637.0,48.0,12.0,0.0,11.0,19.0,18.0


## Missingness and Variance Check

In [31]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [32]:
numeric_features = ['aadt_count', 'dist_to_read_m',
                    'median_household_income', 'households_total', 'total_population',
                    'age_18_29_proportion', 'age_30_64_proportion', 'age_65_plus_proportion',
                    'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
                    'labor_force_participation_rate', 'unemployment_rate',
                    'owner_occupied_proportion', 'renter_occupied_proportion',
                    'family_households_proportion', 'nonfamily_multiperson_households_proportion',
                    'living_alone_households_proportion', 'no_vehicle_households_proportion',
                    'D1A', 'D1B', 'D1C', 'D1C8_RET', 'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT',
                    'D1C8_ED', 'D1C8_HLTH', 'D1C8_PUB', 'D3A', 'NatWalkInd',
                    'gas_station_within_0.5_mi', 'gas_station_within_1_mi', 'arts_and_entertainment_within_0.5_mi',
                    'arts_and_entertainment_within_1_mi', 'shopping_within_0.5_mi',
                    'shopping_within_1_mi', 'social_or_community_service_within_0.5_mi',
                    'social_or_community_service_within_1_mi', 'restaurant_within_0.5_mi',
                    'restaurant_within_1_mi', 'bar_within_0.5_mi', 'bar_within_1_mi',
                    'rac_c000', 'rac_ce01', 'rac_ce02', 'rac_ce03', 'wac_c000', 'wac_cns07',
                    'wac_cns18', 'wac_ce01', 'wac_ce02', 'wac_ce03']

categorical_features = ['chain', 'bustype', 'aadt_functional_class', 'aadt_lanes']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
aadt_count,58.085828,0.000000,3.180915,5466.000000,5865.659472,8804.543594,5334.000000,1.501032,563
dist_to_read_m,56.966847,0.000000,0.082988,627.332754,778.900103,445.966603,744.678223,0.572559,6606
D1A,18.314386,0.261889,0.320606,0.000000,17.706218,28.001488,22.008052,1.581449,6444
D1B,18.314386,0.202369,0.247741,0.000000,42.559408,64.675765,56.678018,1.519658,6450
D1C,18.314386,0.136897,0.167590,0.000000,30.103823,140.466582,11.276514,4.666071,6455
D1C8_RET,18.314386,4.821142,5.902069,0.000000,2.599771,10.403593,1.586033,4.001734,5926
D1C8_PUB,18.314386,66.728171,81.689012,0.000000,1.397457,20.067523,0.000000,14.360026,1000
D1C8_HLTH,18.314386,12.237367,14.981055,0.000000,4.933529,26.571133,1.387777,5.385827,5274
D1C8_ED,18.314386,40.955895,50.138444,0.000000,1.675417,12.355113,0.294076,7.374353,2937
D1C8_ENT,18.314386,9.713708,11.891577,0.000000,3.077399,14.310108,1.099186,4.650066,5432


In [54]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':              pct
 chain           
 0      74.822927
 30      3.345039
 23      2.208202
 46      1.720136
 296     1.410630
 ...          ...
 56      0.005952
 336     0.005952
 337     0.005952
 303     0.005952
 103     0.005952
 
 [106 rows x 1 columns]}

## Correlation

In [38]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951
